In [1]:
import pandas as pd
import numpy as np

# Task 0
Data extraction: get the data from 3 tables & combine it into single `.csv` file.
After that read this file using pandas to create Dataframe.
So it will be all joined data in 1 dataframe. Quick check - should be 74818 rows in it.

In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3


sns.set_theme(style="whitegrid")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [37]:
conn = sqlite3.connect("../db.sqlite3")

df_orderitem = pd.read_sql_query("SELECT* FROM restaurant_orderitem", conn)
df_order = pd.read_sql_query("SELECT* FROM restaurant_order", conn)
df_product = pd.read_sql_query("SELECT* FROM restaurant_product", conn)

df = pd.merge(left=df_orderitem, right=df_order, left_on="order_id", right_on="id")
df = pd.merge(left=df, right=df_product, left_on="product_id", right_on="id")
df = df.drop(columns=["id_x", "id_y", "id"])

df["datetime"] = pd.to_datetime(df["datetime"])
df["total_price"] = df["quantity"] * df["price"]

conn.close()

df.to_csv("restaurant.csv", index=False)

print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74818 entries, 0 to 74817
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   quantity     74818 non-null  int64         
 1   order_id     74818 non-null  int64         
 2   product_id   74818 non-null  int64         
 3   datetime     74818 non-null  datetime64[ns]
 4   price        74818 non-null  float64       
 5   name         74818 non-null  object        
 6   total_price  74818 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(3), object(1)
memory usage: 4.0+ MB
None


# Task 1
Get Top 10 most popular products in restaurant sold by Quantity.
Count how many times each product was sold and create a pie chart with percentage of popularity (by quantity) for top 10 of them.

Example:

![pie chart](../demo/pie.png)

In [ ]:
product_quantities = df.groupby("name")["quantity"].sum().sort_values(ascending=False)
top_10_products = product_quantities[:10]

fig, ax = plt.subplots()

ax.pie(x=top_10_products, labels=top_10_products.index, autopct="%1.1f%%", pctdistance=0.8)

plt.show()

# Task 2
Calculate `Item Price` (Product Price * Quantity) for each Order Item in dataframe.
And Make the same Top 10 pie chart, but this time by `Item Price`. So this chart should describe not the most popular products by quantity, but which products (top 10) make the most money for restaurant. It should be also with percentage.

In [ ]:
item_prices = df.groupby("name")["total_price"].sum().sort_values(ascending=False)
top_10_products = item_prices[:10]

fig, ax = plt.subplots()

ax.pie(x=top_10_products, labels=top_10_products.index, autopct="%1.2f%%", pctdistance=0.8)

plt.show()

# Task 3
Calculate `Order Hour` based on `Order Datetime`, which will tell about the specific our the order was created (from 0 to 23). Using `Order Hour` create a bar chart, which will tell the total restaurant income based on the hour order was created. So on x-axis - it will be values from 0 to 23 (hours), on y-axis - it will be the total sum of order prices, which were sold on that hour.

Example:

![bar chart](../demo/bar.png)

In [ ]:
profit_by_hours = df.groupby(df["datetime"].dt.hour)["total_price"].sum()

fig, ax = plt.subplots(figsize=(10, 4))

ax.bar(profit_by_hours.index, profit_by_hours.values)

ax.set_xlabel("Hour", fontsize=10, weight="bold")
ax.set_ylabel("Sum prices", fontsize=10, weight="bold")
ax.set_xticks(np.arange(0, 25))

plt.show()

# Task 4
Make similar bar chart, but right now with `Order Day Of The Week` (from Monday to Sunday), and also analyze total restaurant income by each day of the week.

In [ ]:
profit_by_weekday = df.groupby(df["datetime"].dt.day_name())["total_price"].sum()
day_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]
profit_by_weekday = profit_by_weekday.reindex(day_order)

fig, ax = plt.subplots()

ax.bar(profit_by_weekday.index, profit_by_weekday.values)

ax.set_xlabel("Day of the week", fontsize=10, weight="bold")
ax.set_ylabel("Sum prices", fontsize=10, weight="bold")

plt.show()